# Milestone 3: the event study

This notebook imports `earnings_vol` and defines no logic of its own. The walkthrough is `docs/milestones/03-event-study.md`; the decisions are ADR 0004 and ADR 0005.

**The timing labels are provisional**: 20 press releases were looked up and every other event keeps the calendar's label until SEC EDGAR is set up. Every result below depends on them.

To rebuild (it needs the local cache, and runs gate G1 before any result):

```
uv run python -m earnings_vol.panel.run
uv run python -m earnings_vol.stats.run --g1
uv run python -m earnings_vol.stats.run
uv run python tools/export_app_data.py
```

Contract schema version: `0.4.0`.

## 1. Abnormal log IV, market-adjusted (ADR 0004 §2)
Made-up numbers: ln IV x = 0.10, 0.12, 0.11, 0.30 with the first three sessions the baseline, ln VIX m = 0, 0.1, 0, 0.2 and beta = 0.5. ALIV = (x - mean_B x) - beta (m - mean_B m); the mean-adjusted twin drops the VIX term.

In [ ]:
import numpy as np

from earnings_vol.panel import abnormal

x = np.array([[0.10, 0.12, 0.11, 0.30]])
m = np.array([[0.0, 0.1, 0.0, 0.2]])
b = np.array([[True, True, True, False]])
aliv, aliv0 = abnormal.aliv(x, m, b, np.array([0.5]))
print("ALIV  ", aliv.round(4))
print("ALIV0 ", aliv0.round(4))
print("e     ", abnormal.abnormal_changes(x, m, np.array([0.5])).round(4))


## 2. The span-change identity (ADR 0005 §1)
Baseline B = {0, 1, 3}, usable sessions {0, 1, 3, 4, 6}, target 6. The walk's changes carry weight n/|B|, n = baseline sessions before the change's end; their sum is ALIV at the target exactly, gaps included.

In [ ]:
in_b = np.array([1, 1, 0, 1, 0, 0, 0], bool)
usable = np.array([1, 1, 0, 1, 1, 0, 1], bool)
walk = abnormal.span_changes(in_b, usable, 6)
xs = np.array([0, 1, np.nan, 4, 2, np.nan, 5], float)
print(walk)
print("span sum", abnormal.span_value(walk, xs, np.zeros(7), 0.0), " level", xs[6] - xs[in_b].mean())


## 3. Two-way clustering (ADR 0005 §2b)
Four made-up events on three dates and two firms: V = V_date + V_firm - V_date&firm, each CR1, and t is compared with t_{min(G) - 1}, here t_1.

In [ ]:
from earnings_vol.stats import clustering

r = clustering.twoway(np.array([0.1, 0.3, -0.2, 0.6]), ["d1", "d1", "d2", "d3"], ["F1", "F2", "F1", "F2"])
print({k: round(v, 6) if isinstance(v, float) else v for k, v in r.items()})


## 4. Gate G1 and the primary family (committed `results`)
G1's calibrated design is binding: each primary statistic must reject at most 6.30% of 2,000 null replications. Then the primary rows, with Holm over the seven declared hypotheses (H5-H7 enter as p = 1 until milestone 4). H2's p sits on the rad_season_p floor 2/2^G, so its Holm verdict depends on the number of season clusters G: the season-cut row (the last season merged into the one before, G - 1) prints the Holm p it would get instead.

A robustness row whose statistic G1 finds oversized keeps its estimate and loses its p-value and interval (ADR 0005 §4). Oversized means above the bar for the replications its rate was measured on: 6.30% at 2,000, 5.57% at 10,000. A rate within one point of 6.30% on 2,000 replications is re-measured on 10,000 first (the `g1:calibrated_remeasured` rows).

In [ ]:
import pandas as pd

from earnings_vol import params, schema, store

derived = params.ROOT / "data" / "derived"
res = store.read_table(derived / "results.parquet", schema.RESULTS)
g1 = res[(res["family"] == "validation") & (res["measure"] == "g1:calibrated") & (res["role"] == "primary")]
print(g1[["hypothesis", "statistic", "n", "estimate"]].to_string(index=False))
cols = ["hypothesis", "measure", "statistic", "n", "estimate", "se", "stat", "df", "p_value", "p_holm"]
prim = res[(res["family"] == "event_test") & (res["hypothesis"].notna()) & (res["adjustment"] == "market")
           & (res["role"] == "primary")]
with pd.option_context("display.width", 160):
    print(prim[cols].to_string(index=False))
cut = res[(res["family"] == "event_test") & (res["subgroup_key"] == "season_cut")]
print(cut[["hypothesis", "subgroup_value", "n_clusters", "p_value", "p_holm"]].to_string(index=False))
rem = res[(res["family"] == "validation") & (res["measure"] == "g1:calibrated_remeasured")]
print(rem[["hypothesis", "statistic", "n", "estimate"]].to_string(index=False))
held = res[(res["family"] == "event_test") & (res["role"] == "robustness") & res["p_value"].isna()
           & (res["adjustment"] == "market") & (res["subgroup_key"] == "all")]
print(held[["hypothesis", "statistic", "estimate", "stat"]].to_string(index=False))


## 5. The mean abnormal path (descriptive)
Pointwise 95% bands, two-way clustered by season and company (ADR 0004 §3): a description, not a test of each day.

In [ ]:
path = res[(res["measure"] == "aliv_path") & (res["subgroup_key"] == "all")]
print(path.pivot(index="rel_day", columns="adjustment", values="estimate").round(4).to_string())
